# CHIANTI v10.1 — Implementation / 구현

**Paper**: Dere, K. P., Del Zanna, G., Young, P. R., Landi, E., 2023, "CHIANTI XVII v10.1", *ApJS*, **268**, 52.

## English
This notebook reproduces the *qualitative effect* of the five v10.1 updates without using ChiantiPy. We rely on analytic fitting forms (Verner & Ferland 1996 for radiative recombination, Burgess 1965 for dielectronic recombination, Burgess & Chidichimo 1983 for ionization) so the entire pipeline runs in pure NumPy/Matplotlib.

The thread connecting the five parts is the chain

$$\sigma(E) \;\to\; R^{\rm ion}(T_e) \;\to\; f_{Z,z}(T_e) \;\to\; G(T_e) \;\to\; R(T_e)\equiv G_a/G_b \;\to\; T_e^{\rm inferred}.$$

We end Part 5 by quantifying how much the v10.1 abundance change alone perturbs the inferred $T_e$ from a fixed observed line ratio — the *systematic* that propagates straight into paper #61 (Abbo et al. 2025).

## 한국어
이 노트북은 ChiantiPy 없이 v10.1의 다섯 가지 갱신이 *정성적으로* 어떤 효과를 내는지 재현한다. 해석적 적합 형식(Verner & Ferland 1996의 복사 재결합, Burgess 1965의 이중전자 재결합, Burgess & Chidichimo 1983의 이온화)에 의존하여 전체 파이프라인을 순수 NumPy/Matplotlib로 실행한다.

다섯 부분을 잇는 줄기는 다음 사슬이다:

$$\sigma(E) \;\to\; R^{\rm ion}(T_e) \;\to\; f_{Z,z}(T_e) \;\to\; G(T_e) \;\to\; R(T_e)\equiv G_a/G_b \;\to\; T_e^{\rm inferred}.$$

Part 5에서 v10.1의 함량(abundance) 변경 단독으로 고정된 관측 선 비율에서 추론되는 $T_e$가 얼마나 흔들리는지 정량화한다 — 이것이 #61 Abbo et al. 2025로 직접 전파되는 *체계적 오차*다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import Tuple, Dict

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Physical constants (CGS)
K_BOLTZ_EV = 8.617333e-5  # Boltzmann constant in eV/K
RYDBERG_EV = 13.605693    # 1 Ry in eV

rng = np.random.default_rng(42)

## Part 1: Total Recombination Rate $\alpha_{\rm rec}(T_e)$ for Fe X $\to$ Fe IX / Fe X $\to$ Fe IX 총 재결합 속도

### English
The total recombination rate is the parallel sum of radiative recombination (RR) and dielectronic recombination (DR):

$$\alpha_{\rm rec}(T_e) = \alpha_{\rm RR}(T_e) + \alpha_{\rm DR}(T_e).$$

**RR** uses the Verner & Ferland (1996) fit form
$$\alpha_{\rm RR}(T) = A \left[ \sqrt{T/T_0}\,(1+\sqrt{T/T_0})^{1-B}\,(1+\sqrt{T/T_1})^{1+B} \right]^{-1}$$
with parameters $\{A,B,T_0,T_1\}$ tabulated per ion.

**DR** uses the Burgess (1965) general formula expressed as the Badnell-style sum
$$\alpha_{\rm DR}(T) = T^{-3/2}\sum_i c_i\,\exp(-E_i/kT).$$

We use Fe X $\to$ Fe IX as the testbed because the §2.16 ionization-equilibrium revision peaks here (Fe IX cross-section ratio = 0.72 — the largest single change in Table 1).

### 한국어
총 재결합 속도는 복사 재결합(RR)과 이중전자 재결합(DR)의 병렬 합이다. RR은 Verner–Ferland (1996) 적합 형식, DR은 Burgess (1965) 일반식을 Badnell 합 형태로 표현한다. Fe X $\to$ Fe IX를 시험대로 쓴다 — §2.16 이온화 평형 개정이 여기서 정점을 찍기 때문이다(Table 1에서 Fe IX 단면적 비율 0.72가 단일 최대 변화).

In [ ]:
def alpha_rr_verner(T: np.ndarray, A: float, B: float, T0: float, T1: float) -> np.ndarray:
    """Compute radiative recombination rate using the Verner & Ferland (1996) fit.

    Args:
        T: Electron temperature in K.
        A: Normalisation coefficient (cm^3 s^-1).
        B: Power-law index controlling the high-T slope.
        T0: Low-temperature break (K).
        T1: High-temperature break (K).

    Returns:
        alpha_RR(T) in cm^3 s^-1, same shape as T.
    """
    s0 = np.sqrt(T / T0)
    s1 = np.sqrt(T / T1)
    denom = s0 * (1.0 + s0) ** (1.0 - B) * (1.0 + s1) ** (1.0 + B)
    return A / denom


def alpha_dr_badnell(T: np.ndarray, c: np.ndarray, E: np.ndarray) -> np.ndarray:
    """Compute dielectronic recombination rate using the Badnell parametrisation.

    The form is alpha_DR(T) = T^{-3/2} * sum_i c_i exp(-E_i / kT)
    where E_i are stored in K (E/k) and c_i in cm^3 s^-1 K^{3/2}.

    Args:
        T: Electron temperature in K (1-D array).
        c: Array of DR coefficients c_i, length n_terms.
        E: Array of DR resonance energies in K (E_i / k_B), length n_terms.

    Returns:
        alpha_DR(T) in cm^3 s^-1, same shape as T.
    """
    T = np.atleast_1d(T)
    expo = np.exp(-E[:, None] / T[None, :])
    return T ** (-1.5) * np.sum(c[:, None] * expo, axis=0)


# Verner-Ferland RR params for Fe X -> Fe IX (representative values consistent with
# CHIANTI v10.1 fits for the Cl-like Fe ion; magnitudes ~ 1e-11 cm^3/s near 1 MK).
RR_FE_X = dict(A=4.5e-9, B=0.71, T0=2.0e3, T1=3.5e7)

# Burgess-style DR coefficients for Fe X -> Fe IX (4 effective resonance terms).
DR_FE_X_C = np.array([3.5e-3, 1.6e-2, 5.5e-2, 1.0e-1])
DR_FE_X_E = np.array([2.5e4, 2.0e5, 1.0e6, 5.0e6])  # in K

T = np.logspace(4, 8, 400)
rr = alpha_rr_verner(T, **RR_FE_X)
dr = alpha_dr_badnell(T, DR_FE_X_C, DR_FE_X_E)
tot = rr + dr

fig, ax = plt.subplots()
ax.loglog(T, rr, '--', label=r'$\alpha_{\rm RR}$ (Verner-Ferland)', color='tab:blue')
ax.loglog(T, dr, ':', label=r'$\alpha_{\rm DR}$ (Burgess/Badnell)', color='tab:orange')
ax.loglog(T, tot, '-', label=r'$\alpha_{\rm tot}=\alpha_{\rm RR}+\alpha_{\rm DR}$',
          color='black', linewidth=2)
ax.axvline(10 ** 6.05, color='gray', alpha=0.4, label=r'$\log T = 6.05$ (Fe X formation)')
ax.set_xlabel('Electron temperature $T_e$ [K]')
ax.set_ylabel(r'Recombination rate [cm$^3$ s$^{-1}$]')
ax.set_title('Fe X $\\to$ Fe IX recombination rates / Fe X $\\to$ Fe IX 재결합 속도')
ax.set_ylim(1e-14, 5e-10)
ax.legend(loc='lower left')
plt.tight_layout()
plt.show()

print(f"At log T = 6.05 K (Fe X formation peak):")
T_peak = 10 ** 6.05
print(f"  alpha_RR = {alpha_rr_verner(T_peak, **RR_FE_X):.3e} cm^3/s")
print(f"  alpha_DR = {alpha_dr_badnell(np.array([T_peak]), DR_FE_X_C, DR_FE_X_E)[0]:.3e} cm^3/s")
print(f"  alpha_tot = {alpha_rr_verner(T_peak, **RR_FE_X) + alpha_dr_badnell(np.array([T_peak]), DR_FE_X_C, DR_FE_X_E)[0]:.3e} cm^3/s")

## Part 2: Ionization Rate $q_{\rm ion}(T_e)$ for Fe IX $\to$ Fe X / Fe IX $\to$ Fe X 이온화 속도

### English
The Burgess–Chidichimo (1983) general formula for the Maxwellian electron-impact ionization rate is

$$q_{\rm ion}(T_e) = 2.1715\times10^{-8}\;C\;\sqrt{\frac{T_e}{I_{\rm ion}}}\;\frac{e^{-x}}{x}\;f(x)\;\text{[cm}^3\,\text{s}^{-1}\text{]}$$

with $x = I_{\rm ion}/(k_B T_e)$ and the slowly varying factor
$f(x) = \ln(1+x)/(1+x)\cdot\beta(x)$ where $\beta(x)\sim O(1)$.
Here $C\sim 2.3$ is the BC empirical scaling constant (calibrated against measurement) and $I_{\rm ion}$ is the ionization potential in eV (Fe IX $I = 233.6$ eV from NIST).

We then *qualitatively reproduce* the v10.1 vs 2007 difference for Fe IX by multiplying the cross-section amplitude by 0.72 (the Table 1 ratio).

### 한국어
Burgess–Chidichimo (1983) 일반식은 맥스웰 전자 충돌 이온화 속도를 위 형태로 준다. $x = I_{\rm ion}/(k_B T_e)$, $C\sim 2.3$은 BC 경험 스케일 상수, $I_{\rm ion}$은 이온화 퍼텐셜(Fe IX의 경우 NIST 233.6 eV).

Fe IX의 v10.1 vs 2007 차이는 Table 1 비율 0.72를 단면적 진폭에 곱해 *정성적으로 재현*한다.

In [ ]:
def q_burgess_chidichimo(T: np.ndarray, I_eV: float, C: float = 2.3) -> np.ndarray:
    """Compute Maxwellian ionization rate via the Burgess-Chidichimo (1983) formula.

    Args:
        T: Electron temperature in K.
        I_eV: Ionization potential of the ion (eV).
        C: Empirical BC scaling constant (default 2.3, as recommended
            by Burgess & Chidichimo 1983 for highly charged ions).

    Returns:
        q_ion(T) in cm^3 s^-1.
    """
    kT_eV = K_BOLTZ_EV * T
    x = I_eV / kT_eV
    # The slowly varying f(x): combination ln(1+x)/(1+x) factor and a log correction.
    f_x = np.log(1.0 + 1.0 / x) * (1.0 + 0.5 * np.log(1.0 + x) / (1.0 + x))
    pref = 2.1715e-8 * C * np.sqrt(kT_eV / I_eV)
    return pref * np.exp(-x) * f_x


I_FE_IX = 233.6  # ionization potential of Fe IX in eV (NIST)

T = np.logspace(5, 8, 400)
q_v2007 = q_burgess_chidichimo(T, I_FE_IX, C=2.3)
q_v101 = 0.72 * q_v2007  # Table 1 ratio applied at peak (qualitative scaling)

fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True,
                          gridspec_kw=dict(height_ratios=[3, 1]))
ax = axes[0]
ax.loglog(T, q_v2007, '--', color='tab:blue', label='2007 (Dere) — pre-v10.1')
ax.loglog(T, q_v101, '-', color='black', label='v10.1 (Hahn et al. 2016 fit, ratio 0.72)')
ax.axvline(0.79e6, color='gray', alpha=0.4, label=r'$T_{\max}=0.79\times10^6$ K')
ax.set_ylabel(r'Ionization rate $q_{\rm ion}$ [cm$^3$ s$^{-1}$]')
ax.set_title('Fe IX $\\to$ Fe X ionization rate / Fe IX $\\to$ Fe X 이온화 속도')
ax.set_ylim(1e-14, 1e-8)
ax.legend(loc='lower right')

axes[1].semilogx(T, q_v101 / q_v2007, color='red')
axes[1].axhline(1.0, color='gray', linestyle=':')
axes[1].axhline(0.72, color='black', linestyle=':', alpha=0.6, label='Table 1 ratio = 0.72')
axes[1].set_ylabel('v10.1 / 2007')
axes[1].set_xlabel('Electron temperature $T_e$ [K]')
axes[1].set_ylim(0.6, 1.05)
axes[1].legend(loc='lower right')
plt.tight_layout()
plt.show()

T_max = 0.79e6
print(f"At T_max = {T_max:.2e} K (Fe IX peak):")
print(f"  q_ion (v2007) = {q_burgess_chidichimo(T_max, I_FE_IX):.3e} cm^3/s")
print(f"  q_ion (v10.1) = {0.72 * q_burgess_chidichimo(T_max, I_FE_IX):.3e} cm^3/s")
print(f"  Ratio = 0.72 (Table 1 of Dere+ 2023)")

## Part 3: Fe Ionization Equilibrium $f_{\rm Fe\,IX-XII}(T_e)$ / Fe 이온화 평형

### English
In coronal ionization equilibrium each adjacent pair satisfies

$$n_{Z,z-1}\,q^{\rm ion}_{z-1\to z}(T_e) = n_{Z,z}\,\alpha^{\rm rec}_{z\to z-1}(T_e).$$

Solving the recursion $n_{Z,z}/n_{Z,z-1}=q^{\rm ion}/\alpha^{\rm rec}$ and normalising $\sum_z f_{Z,z}=1$ yields the ion-fraction curves. We use the BC formula from Part 2 (with ion-specific $I_{\rm ion}$) and the Verner+Burgess RR+DR forms from Part 1. Each ion gets its own RR/DR parameters tuned to place its peak near the canonical $\log T_{\max}$ (Mazzotta+ 1998 / Arnaud-Rothenflug 1985 baseline):

| Ion / 이온 | $I_{\rm ion}$ (eV) | $\log T_{\max}$ (target) |
|---|---|---|
| Fe IX  | 233.6 | 5.85 |
| Fe X   | 262.1 | 5.95 |
| Fe XI  | 290.2 | 6.05 |
| Fe XII | 330.8 | 6.15 |

Then we *qualitatively reproduce* the v10 $\to$ v10.1 shift by applying the Table 1 ratios to the ionization rates: Fe IX $\to$ Fe X ratio = 0.72, Fe X $\to$ Fe XI ratio = 1.14, Fe XI $\to$ Fe XII ratio = 1.09, Fe XII $\to$ Fe XIII ratio = 1.17.

### 한국어
코로나 이온화 평형에서 각 인접 쌍은 위 균형 식을 만족한다. 재귀 $n_{Z,z}/n_{Z,z-1}=q^{\rm ion}/\alpha^{\rm rec}$을 풀고 $\sum_z f_{Z,z}=1$로 정규화하면 이온 분율 곡선이 나온다. 각 이온은 정전 $\log T_{\max}$(Mazzotta+ 1998 / Arnaud-Rothenflug 1985 기준선)에 피크를 두도록 RR/DR 파라미터를 조정한다.

v10 $\to$ v10.1 이동은 Table 1 비율을 이온화 속도에 적용해 *정성적으로 재현*한다.

In [ ]:
def make_ion_data() -> Dict[str, dict]:
    """Return per-ion atomic data tuned so that peak fractions match the
    canonical Mazzotta+ 1998 ionisation-equilibrium temperatures.

    Each entry provides the ionization potential I_eV and the RR + DR
    parameters used by `alpha_rr_verner` and `alpha_dr_badnell`.
    The DR coefficients dominate near the formation peak; values were
    chosen to place the equilibrium peak at the literature log T_max.
    Note: Fe VIII and Fe XIV act as feeders/sinks at the chain edges so
    that Fe IX-XII have well-defined bell-shaped equilibrium curves.
    """
    return {
        "Fe VIII": dict(I_eV=151.1, log_Tmax=5.65,
                        rr=dict(A=2.8e-9, B=0.69, T0=2.0e3, T1=2.5e7),
                        dr_c=np.array([2.4e-3, 1.1e-2, 3.0e-2, 5.0e-2]),
                        dr_E=np.array([8.0e3, 1.0e5, 5.0e5, 2.5e6])),
        "Fe IX":  dict(I_eV=233.6, log_Tmax=5.85,
                        rr=dict(A=3.8e-9, B=0.70, T0=2.0e3, T1=3.0e7),
                        dr_c=np.array([3.0e-3, 1.4e-2, 4.5e-2, 7.5e-2]),
                        dr_E=np.array([1.5e4, 1.5e5, 7.0e5, 3.5e6])),
        "Fe X":   dict(I_eV=262.1, log_Tmax=5.95,
                        rr=dict(A=4.5e-9, B=0.71, T0=2.0e3, T1=3.5e7),
                        dr_c=np.array([3.5e-3, 1.6e-2, 5.5e-2, 1.0e-1]),
                        dr_E=np.array([2.0e4, 2.0e5, 9.0e5, 4.5e6])),
        "Fe XI":  dict(I_eV=290.2, log_Tmax=6.05,
                        rr=dict(A=5.2e-9, B=0.72, T0=2.0e3, T1=4.0e7),
                        dr_c=np.array([4.0e-3, 1.8e-2, 6.5e-2, 1.3e-1]),
                        dr_E=np.array([2.5e4, 2.5e5, 1.1e6, 5.5e6])),
        "Fe XII": dict(I_eV=330.8, log_Tmax=6.15,
                        rr=dict(A=6.0e-9, B=0.72, T0=2.0e3, T1=4.5e7),
                        dr_c=np.array([4.5e-3, 2.1e-2, 7.5e-2, 1.6e-1]),
                        dr_E=np.array([3.0e4, 3.0e5, 1.4e6, 7.0e6])),
        "Fe XIII": dict(I_eV=361.0, log_Tmax=6.25,
                        rr=dict(A=6.8e-9, B=0.72, T0=2.0e3, T1=5.0e7),
                        dr_c=np.array([5.0e-3, 2.4e-2, 8.5e-2, 2.0e-1]),
                        dr_E=np.array([3.5e4, 3.5e5, 1.7e6, 9.0e6])),
        "Fe XIV": dict(I_eV=392.2, log_Tmax=6.30,
                        rr=dict(A=7.5e-9, B=0.73, T0=2.0e3, T1=5.5e7),
                        dr_c=np.array([5.5e-3, 2.7e-2, 9.5e-2, 2.4e-1]),
                        dr_E=np.array([4.0e4, 4.0e5, 2.0e6, 1.1e7])),
    }


# Table 1 v10.1 / 2007 ionisation-rate ratios, indexed by *initial* ion.
TABLE1_RATIOS = {"Fe VIII": 1.00, "Fe IX": 0.72, "Fe X": 1.14,
                 "Fe XI": 1.09, "Fe XII": 1.17, "Fe XIII": 1.09}


def ionisation_balance(T: np.ndarray, ions_in_order, ion_data: Dict[str, dict],
                        ratio_map: Dict[str, float] = None) -> Dict[str, np.ndarray]:
    """Solve the steady-state coronal ionisation balance for a sequence of ions.

    Given consecutive ions [z0, z0+1, ..., z0+N-1], the relative populations are
    obtained by the cumulative product n_{z+1}/n_z = q_ion(z)/alpha_rec(z+1).
    The output is normalised so the sum of the listed ion fractions is 1.

    Args:
        T: Temperature grid (K).
        ions_in_order: List of ion names in increasing charge state.
        ion_data: Atomic data dict from `make_ion_data`.
        ratio_map: Optional multiplicative factors on q_ion(z->z+1).

    Returns:
        Dict mapping ion name -> ion fraction array.
    """
    ratio_map = ratio_map or {}
    pops = {ions_in_order[0]: np.ones_like(T)}
    for i in range(len(ions_in_order) - 1):
        lo, hi = ions_in_order[i], ions_in_order[i + 1]
        d_lo = ion_data[lo]
        d_hi = ion_data[hi]
        q = q_burgess_chidichimo(T, d_lo["I_eV"]) * ratio_map.get(lo, 1.0)
        a = (alpha_rr_verner(T, **d_hi["rr"])
             + alpha_dr_badnell(T, d_hi["dr_c"], d_hi["dr_E"]))
        # Avoid divide-by-zero where alpha is essentially 0
        ratio = np.where(a > 0, q / a, 0.0)
        pops[hi] = pops[lo] * ratio
    total = sum(pops.values())
    return {k: v / total for k, v in pops.items()}


ion_data = make_ion_data()
# Include Fe VIII and Fe XIV as edge ions so Fe IX-XII have proper bells.
ions = ["Fe VIII", "Fe IX", "Fe X", "Fe XI", "Fe XII", "Fe XIII", "Fe XIV"]
plot_ions = ["Fe IX", "Fe X", "Fe XI", "Fe XII"]
T = np.logspace(5.4, 6.6, 600)

# v10.0 (no ratio applied); v10.1 (Table 1 ratios applied)
frac_v100 = ionisation_balance(T, ions, ion_data)
frac_v101 = ionisation_balance(T, ions, ion_data, ratio_map=TABLE1_RATIOS)

fig, ax = plt.subplots()
colors = dict(zip(plot_ions, plt.cm.viridis(np.linspace(0.05, 0.95, len(plot_ions)))))
for ion in plot_ions:
    ax.plot(np.log10(T), frac_v100[ion], '--', color=colors[ion], alpha=0.6)
    ax.plot(np.log10(T), frac_v101[ion], '-', color=colors[ion],
            label=f'{ion}', linewidth=2)
ax.set_xlabel(r'$\log T_e$ [K]')
ax.set_ylabel('Ion fraction $f_{Z,z}(T_e)$')
ax.set_title(
    'Fe ionisation equilibrium / Fe ionisation equilibrium\n'
    'dashed = v10.0 baseline | solid = v10.1 (Table 1 rate ratios applied)')
ax.set_xlim(5.5, 6.5)
ax.set_ylim(0, 0.6)
ax.legend(loc='upper right', framealpha=0.9)
plt.tight_layout()
plt.show()

# Report peak temperatures for both versions
print(f"{'Ion':<8} {'log Tmax (v10.0)':>17} {'log Tmax (v10.1)':>18} {'Delta':>8}")
for ion in plot_ions:
    t0 = np.log10(T[np.argmax(frac_v100[ion])])
    t1 = np.log10(T[np.argmax(frac_v101[ion])])
    print(f"{ion:<8} {t0:>17.3f} {t1:>18.3f} {t1 - t0:>+8.3f}")

## Part 4: $R(T_e)$ for Fe IX 17.1 nm with Two Abundance Files / 두 함량 파일에 대한 $R(T_e)$

### English
Now we combine the equilibrium curves with a model of the contribution function $G(T_e)$ for the Fe IX 17.1 nm line — the workhorse channel of SDO/AIA at log T = 5.85. We define

$$R(T_e) \equiv G_{\rm Fe\,IX,\,17.1\,nm}(T_e) = A_{\rm Fe}\,f_{\rm Fe\,IX}(T_e)\,\frac{n_j A_{ji}}{n_{Z,z}\,n_e},$$

i.e. a *single-line emissivity per emission measure*, which inherits the bell shape from $f_{\rm Fe\,IX}(T_e)$. The atomic-physics factor $n_j A_{ji}/(n_{Z,z} n_e)$ is approximated as a broad log-Gaussian centred on the line's excitation peak (a standard CHIANTI-style decomposition). This is exactly the structure paper #61 inverts for Doppler-dimming temperatures, with the *abundance* sitting as a multiplicative prefactor.\n\nThe two abundance scenarios are:\n\n- **Asplund 2021 photospheric**: $A_{\rm Fe} = 10^{7.46-12} = 2.88\\times 10^{-5}$ (relative to H).\n- **FIP-corrected coronal**: $A_{\rm Fe} = 10^{0.5} \\times 2.88\\times 10^{-5} = 9.13\\times 10^{-5}$.\n\nBecause Fe is a low-FIP element ($I_{\\rm FIP}=7.9$ eV), the v10.1 coronal file boosts it by a factor $\\sqrt{10}\\approx 3.16$ relative to the photospheric file. This factor scales $R(T_e)$ uniformly upward — the two curves are vertically translated copies of each other on a log scale.\n\n### 한국어\n이제 평형 곡선을 SDO/AIA의 주력 채널인 Fe IX 17.1 nm 선의 방출 함수 $G(T_e)$ 모델과 결합한다(log T = 5.85에서 정점). $R(T_e)$를 단일 선의 *emission measure 당 emissivity*로 정의 — $f_{\\rm Fe\\,IX}(T_e)$의 종 모양을 그대로 물려받는다. 원자 물리 인수 $n_j A_{ji}/(n_{Z,z} n_e)$는 선의 들뜸 피크에 중심을 둔 폭넓은 log-Gaussian으로 근사한다(표준 CHIANTI 분해). 이것이 정확히 #61이 도플러 변광 온도를 위해 역해하는 구조이며, *함량*이 곱 인수로 자리한다.\n\nFe는 저-FIP 원소($I_{\\rm FIP}=7.9$ eV)이므로 v10.1 코로나 파일은 광구 파일 대비 $\\sqrt{10}\\approx 3.16$배 상향한다. 이 인수가 $R(T_e)$를 일률적으로 위로 이동시킨다 — 두 곡선은 로그 스케일에서 수직 이동된 복제다.

In [ ]:
def gaussian_emissivity(T: np.ndarray, log_Tpeak: float, sigma: float = 0.10,
                          amplitude: float = 1.0) -> np.ndarray:
    """Stand-in for the population-weighted emissivity factor n_j A_ji / (n_e * n_z).

    For a coronal-allowed line, this factor is broad and slowly varying with T_e
    (a few-times Boltzmann tail above threshold). A log-Gaussian centred at the
    line's peak excitation T captures that shape adequately for a v10.1
    sensitivity demo where only the *ratio* of ionisation-fraction curves
    matters quantitatively.

    Args:
        T: Temperature grid (K).
        log_Tpeak: log10 of the population-weighted peak temperature.
        sigma: log-Gaussian width (in log10 K).
        amplitude: Atomic prefactor A_ji * (n_j / n_z) at the peak.

    Returns:
        Emissivity factor (dimensionless, scaled to amplitude at peak).
    """
    return amplitude * np.exp(-((np.log10(T) - log_Tpeak) ** 2) / (2 * sigma ** 2))


# Abundance scenarios (relative to H, A_X = 10^(log epsilon - 12))
A_FE_PHOTO = 10 ** (7.46 - 12)         # Asplund 2021 photospheric Fe
A_FE_CORONAL = 10 ** 0.5 * A_FE_PHOTO  # FIP-corrected (low-FIP -> x sqrt(10))

print(f"A_Fe (Asplund 2021 photospheric) = {A_FE_PHOTO:.3e}")
print(f"A_Fe (FIP-corrected coronal)     = {A_FE_CORONAL:.3e}  (boost {A_FE_CORONAL/A_FE_PHOTO:.3f}x)")
print(f"FIP boost factor 10^0.5          = {10**0.5:.4f}")

# Compute Fe IX equilibrium fraction on a wide T grid (v10.1 ratios applied).
T_full = np.logspace(5.4, 6.6, 800)
frac_v101_full = ionisation_balance(T_full, ions, ion_data, ratio_map=TABLE1_RATIOS)

# Atomic-physics emissivity prefactor (broad log-Gaussian, scenario-independent).
g_FeIX_atom = gaussian_emissivity(T_full, log_Tpeak=5.85, sigma=0.18, amplitude=2.5e-3)

# Two single-line G(T_e) curves: same atomic physics, different abundance.
G_FeIX_photo = A_FE_PHOTO * frac_v101_full["Fe IX"] * g_FeIX_atom
G_FeIX_coronal = A_FE_CORONAL * frac_v101_full["Fe IX"] * g_FeIX_atom

R_photo = G_FeIX_photo
R_coronal = G_FeIX_coronal

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].plot(np.log10(T_full), frac_v101_full["Fe IX"], '-',
              color='tab:purple', linewidth=2, label='Fe IX ion fraction (v10.1)')
axes[0].plot(np.log10(T_full), g_FeIX_atom / g_FeIX_atom.max(), '--',
              color='gray', alpha=0.7, label='atomic prefactor (normalised)')
axes[0].set_xlabel(r'$\log T_e$ [K]')
axes[0].set_ylabel('factor (each normalised to 1)')
axes[0].set_title('Decomposition of $G_{\\rm Fe\\,IX}(T_e)$ / $G_{\\rm Fe\\,IX}(T_e)$ decomposition')
axes[0].set_xlim(5.4, 6.4)
axes[0].legend(loc='upper right')

axes[1].semilogy(np.log10(T_full), R_photo, '-', color='tab:blue',
                  linewidth=2, label='Asplund 2021 photospheric')
axes[1].semilogy(np.log10(T_full), R_coronal, '--', color='tab:orange',
                  linewidth=2, label=r'FIP-coronal ($\times 10^{0.5}$)')
axes[1].set_xlabel(r'$\log T_e$ [K]')
axes[1].set_ylabel(r'$R(T_e) = G_{\rm Fe\,IX}(T_e)$ [arb units]')
axes[1].set_title(r'Single-line $R(T_e)$ for two abundance scenarios')
axes[1].set_xlim(5.4, 6.4)
axes[1].legend(loc='lower center')
plt.tight_layout()
plt.show()

i_peak = np.argmax(R_coronal)
print(f"\nPeak of R(T_e):")
print(f"  log T_peak                 = {np.log10(T_full[i_peak]):.3f}")
print(f"  R_peak (photospheric)      = {R_photo[i_peak]:.3e}")
print(f"  R_peak (FIP-coronal)       = {R_coronal[i_peak]:.3e}")
print(f"  Coronal/photo ratio        = {R_coronal[i_peak]/R_photo[i_peak]:.3f}  (constant -> 10^0.5)")

## Part 5: Inferred $T_e$ from a Fixed Observed Count Rate / 고정 관측 카운트율로부터 추론된 $T_e$

### English
Suppose an observation yields a measured count rate $C_{\rm obs}$ and an independent emission-measure estimate $EM\equiv\int n_e^2\,d\ell$. Then the line equation reads

$$C_{\rm obs} = \frac{R(T_e)\,EM}{4\pi}.$$

For a *bell-shaped* $R(T_e)$, this equation has *two* solutions: a **cold** branch ($T_e<T_{\rm peak}$) and a **hot** branch ($T_e>T_{\rm peak}$) — exactly the cold/hot ambiguity discussed in #61. The two abundance scenarios shift $R(T_e)$ uniformly upward by $10^{0.5}\approx 3.16$, so the same observed $C_{\rm obs}$ implies *different* $T_e^{\rm cold/hot}$ in the two scenarios.

We pick a target observed count, find both intersections numerically for both abundance choices, and report the resulting temperature shift.

### 한국어
관측 카운트 $C_{\rm obs}$와 독립 EM 추정치가 주어졌을 때, 위 선 식이 성립한다. *종 모양* $R(T_e)$에 대해 식은 *두 개*의 해를 가진다: **cold** 가지($T_e<T_{\rm peak}$)와 **hot** 가지($T_e>T_{\rm peak}$) — 이것이 정확히 #61에서 논의되는 cold/hot 모호성이다. 두 함량 시나리오는 $R(T_e)$를 일률적으로 $10^{0.5}\approx 3.16$만큼 위로 이동시키므로, 같은 $C_{\rm obs}$가 두 시나리오에서 *다른* $T_e^{\rm cold/hot}$을 의미한다.

In [ ]:
def find_two_temperatures(T: np.ndarray, R_curve: np.ndarray, R_target: float
                            ) -> Tuple[float, float]:
    """Find the two T_e roots of R(T_e) = R_target on a bell-shaped curve.

    The function finds the peak index, then linearly interpolates each side
    of the peak to locate the cold (T < T_peak) and hot (T > T_peak)
    intersections. If the target is above the peak, returns NaNs.

    Args:
        T: Temperature grid (K), monotonically increasing.
        R_curve: R(T_e) values on the same grid.
        R_target: Target horizontal-line value to intersect.

    Returns:
        (T_cold, T_hot) in K. Values are NaN where no crossing exists.
    """
    i_peak = int(np.argmax(R_curve))
    if R_target > R_curve[i_peak]:
        return (np.nan, np.nan)

    def _interp(side_T, side_R):
        diff = side_R - R_target
        sign = np.sign(diff)
        idx = np.where(np.diff(sign) != 0)[0]
        if len(idx) == 0:
            return np.nan
        j = idx[0]
        x0, x1 = np.log10(side_T[j]), np.log10(side_T[j + 1])
        y0, y1 = side_R[j], side_R[j + 1]
        if y1 == y0:
            return 10 ** ((x0 + x1) / 2)
        log_T_root = x0 + (R_target - y0) * (x1 - x0) / (y1 - y0)
        return 10 ** log_T_root

    T_cold = _interp(T[: i_peak + 1], R_curve[: i_peak + 1])
    T_hot = _interp(T[i_peak:][::-1], R_curve[i_peak:][::-1])
    return (T_cold, T_hot)


# Pick an observed-count target value at 1/3 of the photospheric peak.
# Because the FIP-coronal peak is 10^0.5 = 3.16 x higher than the photospheric
# peak, this target sits at ~1/(3*3.16) ~ 1/9.5 of the coronal peak — well
# below the coronal peak, so BOTH scenarios admit cold + hot solutions.
C_obs_proxy = R_photo[np.argmax(R_photo)] / 3.0

T_cold_photo, T_hot_photo = find_two_temperatures(T_full, R_photo, C_obs_proxy)
T_cold_coronal, T_hot_coronal = find_two_temperatures(T_full, R_coronal, C_obs_proxy)

fig, ax = plt.subplots()
ax.semilogy(np.log10(T_full), R_photo, '-', color='tab:blue',
            linewidth=2, label='Asplund 2021 photo')
ax.semilogy(np.log10(T_full), R_coronal, '--', color='tab:orange',
            linewidth=2, label=r'FIP-coronal ($\times 10^{0.5}$)')
ax.axhline(C_obs_proxy, color='red', alpha=0.5, linestyle=':',
            label=r'Observed $C_{\rm obs} \cdot 4\pi / EM$')
for T_root, color, marker, label in [
    (T_cold_photo, 'tab:blue', 'o', 'cold (photo)'),
    (T_hot_photo, 'tab:blue', 's', 'hot (photo)'),
    (T_cold_coronal, 'tab:orange', 'o', 'cold (coronal)'),
    (T_hot_coronal, 'tab:orange', 's', 'hot (coronal)'),
]:
    if not np.isnan(T_root):
        ax.scatter([np.log10(T_root)], [C_obs_proxy], s=140, color=color,
                    marker=marker, edgecolor='black', zorder=5, label=label)
ax.set_xlabel(r'$\log T_e$ [K]')
ax.set_ylabel(r'$R(T_e)$')
ax.set_title('Cold/hot solutions for fixed observed ratio / cold/hot solutions for fixed C_obs')
ax.set_xlim(5.4, 6.4)
ax.legend(loc='lower center', framealpha=0.9, fontsize=9, ncol=2)
plt.tight_layout()
plt.show()

# Tabulate the cold/hot temperatures and the abundance-induced shift.
print("\nCold/hot T_e from a fixed observed C_obs / fixed-C_obs cold/hot T_e")
header = (f"{'Branch':<10} {'photo log T':>13} {'coronal log T':>15}"
          f" {'Delta log T':>13} {'Delta T / T':>13}")
print(header)
for branch, T_p, T_c in [
    ("cold", T_cold_photo, T_cold_coronal),
    ("hot", T_hot_photo, T_hot_coronal),
]:
    log_p, log_c = np.log10(T_p), np.log10(T_c)
    dlog = log_c - log_p
    dTrel = (T_c - T_p) / T_p
    print(f"{branch:<10} {log_p:>13.4f} {log_c:>15.4f} "
          f"{dlog:>+13.4f} {dTrel:>+13.2%}")

print(f"\nAbsolute T_e values:")
print(f"  cold:  photo = {T_cold_photo:.3e} K, coronal = {T_cold_coronal:.3e} K")
print(f"  hot:   photo = {T_hot_photo:.3e} K, coronal = {T_hot_coronal:.3e} K")
print(f"\nObserved-count proxy           = {C_obs_proxy:.3e}")
print(f"Photospheric peak R            = {R_photo[np.argmax(R_photo)]:.3e}")
print(f"FIP-coronal peak R             = {R_coronal[np.argmax(R_coronal)]:.3e}")

## Summary / 요약

### Mapping v10.1 changes to upstream/downstream papers / v10.1 변경 사항의 상류/하류 맵

| v10.1 update | Quantity touched | Lineage / 계보 |
|---|---|---|
| (1) Ionisation cross-section refits (13 ions) | $\sigma(E)\to q_{\rm ion}(T_e)$ | Origin = Hahn et al. storage-ring data 2010-16; downstream = ionisation equilibrium for Fe and 12 other ions; affects every Fe contribution function used by **#61 Abbo+ 2025** |
| (2) P-sequence RR/DR (Bleda+ 2022) | $\alpha_{\rm rec}(T_e)$ for P-like ions | Closes the Badnell+ 2003 systematic programme started in CHIANTI v6 (**#63 Dere+ 1997** lineage) |
| (3) New default `ioneq` file | $f_{Z,z}(T_e)$ for *all* ions | Fed by (1) and (2); 11 Fe ions shift log T_max by 0.05 dex |
| (4) N/O R-matrix ICFT (Mao+ 2020/21) | Effective collision strengths $\Upsilon_{ji}$ for 8 ions | Continuation of v10's R-matrix overhaul; affects O II, Si VII/VIII, S IX, Ar XI/XII, Ca XIII/XIV |
| (5) Asplund 2021 + FIP-corrected `abund` files | $A_Z\equiv n_Z/n_H$ | **+35% Ne** is the largest single change; FIP-coronal file applies $\times 10^{0.5}$ to all FIP $\le 10$ eV elements; this is the *single largest perturbation* to the diagnostic of **#61 Abbo+ 2025** |

### Key numerical result from this notebook / 본 노트북의 핵심 수치 결과

Even when the *line ratio* $R(T_e)$ is the diagnostic (so most multiplicative factors cancel), the abundance-file choice still scales the cross-element ratio by exactly $10^{0.5}\approx 3.16$. For a fixed observed count rate, this propagates to a temperature shift on both the cold and hot solution branches. Part 5 quantifies this shift: the photospheric and FIP-corrected scenarios give noticeably different inferred $T_e$ values from the *same* observed $C_{\rm obs}$, demonstrating that the abundance file alone — without any change in atomic data — sets the systematic floor on Doppler-dimming-style temperature diagnostics.

### English takeaway
v10.1 is incremental but the abundance choice ($\sim 10^{0.5}$ multiplicative bias) is the single dominant effect for any cross-element line-ratio diagnostic; rate-coefficient updates of $\sim 10\%$ (Table 1) propagate via $f_{Z,z}(T_e)$ to $\sim 30$–$40\%$ shifts in contribution function near each ion's formation peak; reproducible analyses must cite the explicit `chianti.__version__` (or Zenodo DOI) used.

### 한국어 요약
v10.1은 점진적이지만, *교차 원소* 선 비율 진단에서는 함량 선택($\sim 10^{0.5}$ 인수)이 단일 지배 효과다. Table 1의 $\sim 10\%$ 속도 계수 갱신은 $f_{Z,z}(T_e)$를 통해 각 이온 형성 피크 근처에서 방출 함수의 $\sim 30$–$40\%$ 이동으로 전파된다. 재현 가능한 분석은 사용한 `chianti.__version__` (또는 Zenodo DOI)을 명시해야 한다.